# Baseline Predictor for FNSPID reduced

# 0. imports

## 1. FNSPID baseline

In [5]:
import pandas as pd
from pathlib import Path

fnspid_path = Path("/home/michaelschlee/ownCloud/GIT/LabelFusion/Dataset_Descriptives/data/FNSPID")
file_2022 = fnspid_path / "train_df.csv"
file_2023 = fnspid_path / "test_df.csv"

if not file_2022.exists():
    print("File not found:", file_2022)
elif not file_2023.exists():
    print("File not found:", file_2023)
else:
    train_df = pd.read_csv(file_2022, dtype="string")
    test_df = pd.read_csv(file_2023, dtype="string")

    print(f"Loaded {len(train_df):,} rows from {file_2022.name}")
    print(f"Loaded {len(test_df):,} rows from {file_2023.name}")

    display(train_df.head())
    display(test_df.head())


Loaded 23,180 rows from train_df.csv
Loaded 5,800 rows from test_df.csv


,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
0,2022-12-31 00:00:00 UTC,Where Will Apple Stock Be in 1 Year?,AAPL,https://www.nasdaq.com/articles/where-will-app...,<NA>,<NA>,Apple's (NASDAQ: AAPL) stock declined nearly 3...
1,2022-12-31 00:00:00 UTC,Where Will Unity Software Stock Be in 3 Years?,AAPL,https://www.nasdaq.com/articles/where-will-uni...,<NA>,<NA>,Unity Software (NYSE: U) attracted a stampede ...
2,2022-12-31 00:00:00 UTC,4 Red-Hot Growth Stocks to Buy in 2023 and Beyond,AAPL,https://www.nasdaq.com/articles/4-red-hot-grow...,<NA>,<NA>,"Inflation, rising interest rates, and other ma..."
3,2022-12-31 00:00:00 UTC,"Even in an Advertising Slowdown, These 3 Stock...",AAPL,https://www.nasdaq.com/articles/even-in-an-adv...,<NA>,<NA>,"Marketers are trimming their ad budgets, and t..."
4,2022-12-31 00:00:00 UTC,3 Unstoppable Growth Stocks to Buy After a Sto...,AAPL,https://www.nasdaq.com/articles/3-unstoppable-...,<NA>,<NA>,The stock market's 2022 sell-off has highlight...


,Date,Article_title,Stock_symbol,Url,Publisher,Author,Article
0,2023-06-13 00:00:00 UTC,3 Surprising EV Stocks Surging on Tesla’s Supe...,TSLA,https://www.nasdaq.com/articles/3-surprising-e...,<NA>,<NA>,"InvestorPlace - Stock Market News, Stock Advic..."
1,2023-10-01 00:00:00 UTC,Nasdaq Sell-Off: My Top 3 Beaten-Down Growth S...,AAPL,https://www.nasdaq.com/articles/nasdaq-sell-of...,<NA>,<NA>,The Nasdaq Composite has come roaring back in ...
2,2023-08-21 00:00:00 UTC,3 Stock-Split Stocks Billionaires Are Selling ...,GOOG,https://www.nasdaq.com/articles/3-stock-split-...,<NA>,<NA>,There are a lot of ways to make money on Wall ...
3,2023-01-01 00:00:00 UTC,Can Nvidia Help Walmart and Target Solve Their...,NVDA,https://www.nasdaq.com/articles/can-nvidia-hel...,<NA>,<NA>,Walmart (NYSE: WMT) and Target (NYSE: TGT) hav...
4,2023-02-06 00:00:00 UTC,Warren Buffett More Than Doubles His Money on ...,BRK,https://www.nasdaq.com/articles/warren-buffett...,<NA>,<NA>,When Berkshire Hathaway (NYSE: BRK.A)(NYSE: BR...


### 1.1 Multinomial Logistic Regression

In [6]:
# TF-IDF + Multinomial Logistic Regression baseline for Stock_symbol
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# locate train/test frames (handles either variable names)
if 'train_df' in globals():
    train = train_df.copy()
elif 'top10_2022' in globals():
    train = top10_2022.copy()
else:
    raise RuntimeError("train dataframe not found (expected `train_df` or `top10_2022`)")

if 'test_df' in globals():
    test = test_df.copy()
elif 'top10_2023' in globals():
    test = top10_2023.copy()
else:
    raise RuntimeError("test dataframe not found (expected `test_df` or `top10_2023`)")

# prepare text feature
train['text'] = train['Article_title'].fillna('').astype(str) + " " + train['Article'].fillna('').astype(str)
test['text']  = test['Article_title'].fillna('').astype(str)  + " " + test['Article'].fillna('').astype(str)

# label encoding
le = LabelEncoder()
y_train = le.fit_transform(train['Stock_symbol'].astype(str))
y_test  = le.transform(test['Stock_symbol'].astype(str))

# TF-IDF
vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train = vec.fit_transform(train['text'])
X_test  = vec.transform(test['text'])

# class weights (balanced)
classes = np.unique(y_train)
cw_vals = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = {int(c): w for c, w in zip(classes, cw_vals)}

# multinomial logistic regression
clf_lr = LogisticRegression(multi_class='multinomial', n_jobs=-1, random_state=42, class_weight=class_weight, max_iter=1000)
clf_lr.fit(X_train, y_train)

# predict + report
pred_lr = clf_lr.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred_lr))
print(classification_report(y_test, pred_lr, target_names=le.classes_))

/home/michaelschlee/ownCloud/GIT/envs/labelFusion/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/lib/python3.12/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)


Accuracy: 0.7182758620689655
              precision    recall  f1-score   support

        AAPL       0.72      0.55      0.62       934
         AMD       0.69      0.68      0.69       349
         BRK       0.81      0.81      0.81       401
         DIS       0.84      0.90      0.87       337
        GOOG       0.51      0.72      0.60       374
        MSFT       0.66      0.64      0.65       980
        NVDA       0.65      0.69      0.67       912
        TSLA       0.81      0.80      0.80       897
         WMT       0.83      0.90      0.86       296
         XOM       0.85      0.88      0.87       320

    accuracy                           0.72      5800
   macro avg       0.74      0.76      0.74      5800
weighted avg       0.72      0.72      0.72      5800



### 1.2 Random Forest

In [7]:
# TF-IDF + RandomForest baseline for Stock_symbol
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

# locate train/test frames (handles either variable names)
if 'train_df' in globals():
    train = train_df.copy()
elif 'top10_2022' in globals():
    train = top10_2022.copy()
else:
    raise RuntimeError("train dataframe not found (expected `train_df` or `top10_2022`)")

if 'test_df' in globals():
    test = test_df.copy()
elif 'top10_2023' in globals():
    test = top10_2023.copy()
else:
    raise RuntimeError("test dataframe not found (expected `test_df` or `top10_2023`)")

# prepare text feature
train['text'] = train['Article_title'].fillna('').astype(str) + " " + train['Article'].fillna('').astype(str)
test['text']  = test['Article_title'].fillna('').astype(str)  + " " + test['Article'].fillna('').astype(str)

# label encoding
le = LabelEncoder()
y_train = le.fit_transform(train['Stock_symbol'].astype(str))
y_test  = le.transform(test['Stock_symbol'].astype(str))

# TF-IDF
vec = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train = vec.fit_transform(train['text'])
X_test  = vec.transform(test['text'])

# class weights (balanced)
classes = np.unique(y_train)
cw_vals = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = {int(c): w for c, w in zip(classes, cw_vals)}

# classifier
clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42, class_weight=class_weight)
clf.fit(X_train, y_train)

# predict + report
pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, target_names=le.classes_))

KeyboardInterrupt: 

### 1.3 XGBoost

In [ ]:
# TF-IDF + XGBoost baseline for Stock_symbol
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# XGBoost classifier with scale_pos_weight for class imbalance handling
clf_xgb = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    tree_method='hist',
    device='cuda' if xgb.get_config()['gpu_id'] != '-1' else 'cpu'
)

clf_xgb.fit(X_train, y_train)

# predict + report
pred_xgb = clf_xgb.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred_xgb))
print(classification_report(y_test, pred_xgb, target_names=le.classes_))

ModuleNotFoundError: No module named 'xgboost'